# A2A Protocol, Powered by Real LLM Agents (Groq)

The previous notebook built the **A2A protocol scaffolding** — Agent Card,
Task lifecycle, JSON-RPC, streaming, orchestration — around toy skill
handlers that just returned hardcoded dicts.

This notebook keeps the exact same scaffolding, but the "skill handlers" are
now genuinely powered by an LLM through the **Groq API** (OpenAI-compatible,
very low latency). The agents actually reason about the input now, instead
of pattern-matching digits out of a string.

You'll build:

1. A `call_groq()` helper wrapping Groq's async chat-completions API
2. An **LLM-backed refund agent** that extracts structured JSON from free text
3. **Real token streaming** relayed live through the A2A SSE stream
4. Two more LLM agents (**summarizer** + **sentiment**) orchestrated in parallel
5. The same auth / idempotency / production notes as before, now with a note on **key hygiene**

> Requires outbound internet access to `api.groq.com` when you actually run the
> LLM cells — this notebook is not pre-executed for that reason (unlike the
> pure-protocol one, which needed no network at all).


## 0. Setup & API Key

Groq keys are provided the same way as any other cloud API key: as an
environment variable, read at runtime, never hardcoded into code you'd commit
or share. The cell below follows exactly that pattern — it only prompts for
a key if one isn't already set.

> **Don't ship this notebook with a real key baked into it.** The line below
> is convenient for a personal, throwaway notebook — in anything shared or
> version-controlled, delete the hardcoded fallback and rely on `getpass`,
> a `.env` file (gitignored), or a secrets manager instead.


In [ ]:
import os
from getpass import getpass

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "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"

# If you'd rather not hardcode anything at all, comment out the block above
# and uncomment this instead — it will prompt you securely, once:
# if "GROQ_API_KEY" not in os.environ:
#     os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

print("GROQ_API_KEY is set:", "GROQ_API_KEY" in os.environ)


In [ ]:
import subprocess, sys

def _pip_install(*pkgs):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *pkgs])
    except subprocess.CalledProcessError:
        # Some environments (e.g. system Python on Debian/Ubuntu) refuse installs
        # unless this flag is passed — harmless everywhere else.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                                "--break-system-packages", *pkgs])

_pip_install("fastapi", "httpx", "pydantic", "uvicorn", "nest_asyncio", "groq")
print("Dependencies installed.")


In [ ]:
import asyncio
import json
import time
import uuid
from enum import Enum
from typing import Any, Dict, List, Optional

import httpx
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field
from groq import AsyncGroq

import nest_asyncio
nest_asyncio.apply()  # lets us use asyncio.run() safely inside Jupyter's own event loop

groq_client = AsyncGroq(api_key=os.environ["GROQ_API_KEY"])

# Groq deprecated llama-3.1-8b-instant / llama-3.3-70b-versatile in mid-2026.
# openai/gpt-oss-20b is the fast/cheap successor; swap to openai/gpt-oss-120b for higher quality.
GROQ_MODEL = "openai/gpt-oss-20b"

print("Groq async client ready. Model:", GROQ_MODEL)


## 1. `call_groq()` — the one function every agent below is built on

Every LLM-backed skill in this notebook goes through this single helper. In
a real codebase this is the seam where you'd add retries, timeouts, token
accounting, and prompt-caching — once, instead of in every agent.


In [ ]:
async def call_groq(system_prompt: str, user_text: str, temperature: float = 0.2) -> str:
    resp = await groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_text},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content


def extract_json(raw: str) -> dict:
    """LLMs often wrap JSON in prose or ```json fences despite instructions
    not to. Try a clean parse first, then fall back to slicing out the
    outermost {...} block — this single fallback fixes the majority of
    real-world 'the model almost followed instructions' failures."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start, end = raw.find("{"), raw.rfind("}")
        return json.loads(raw[start:end + 1])


async def _smoke_test():
    reply = await call_groq(
        "Reply with exactly one short sentence.",
        "Say hello and name one thing agents are good at.",
    )
    print(reply)

asyncio.run(_smoke_test())


## 2. The A2A Vocabulary (same as before)

Unchanged from the protocol notebook — Agent Card, Task, Message, Part,
Artifact. Copy this cell verbatim into any new agent project; it never
changes between agents, only the skill handlers do.


In [ ]:
class TaskState(str, Enum):
    submitted       = "submitted"
    working         = "working"
    input_required  = "input-required"
    auth_required   = "auth-required"
    completed       = "completed"
    failed          = "failed"
    canceled        = "canceled"
    rejected        = "rejected"

TERMINAL_STATES = {TaskState.completed, TaskState.failed, TaskState.canceled, TaskState.rejected}


class Part(BaseModel):
    kind: str
    text: Optional[str] = None
    data: Optional[dict] = None
    file_uri: Optional[str] = None
    mime_type: Optional[str] = None


class Message(BaseModel):
    role: str
    parts: List[Part]
    message_id: str = Field(default_factory=lambda: str(uuid.uuid4())[:8])


class Artifact(BaseModel):
    name: str
    parts: List[Part]


class AgentSkill(BaseModel):
    id: str
    description: str
    input_modes: List[str] = ["text/plain"]
    output_modes: List[str] = ["application/json"]


class AgentCapabilities(BaseModel):
    streaming: bool = False
    push_notifications: bool = False


class AgentCard(BaseModel):
    name: str
    description: str
    url: str
    version: str
    capabilities: AgentCapabilities
    skills: List[AgentSkill]
    security_schemes: Dict[str, dict] = {}


class Task(BaseModel):
    id: str
    context_id: str
    status: TaskState
    history: List[Message] = []
    artifacts: List[Artifact] = []

print("A2A types ready.")


## 3. Server Factory — now with **async, LLM-backed** skill handlers

The only real change from the pure-protocol notebook: `skill_handler` is now
`async def skill_handler(text: str) -> dict`, so it can `await call_groq(...)`
without blocking the event loop while the network round-trip to Groq is in flight.


In [ ]:
class InMemoryTaskStore:
    def __init__(self):
        self._tasks: Dict[str, Task] = {}

    def create(self, context_id: str, first_message: Message) -> Task:
        task = Task(id=str(uuid.uuid4())[:8], context_id=context_id,
                     status=TaskState.submitted, history=[first_message])
        self._tasks[task.id] = task
        return task

    def get(self, task_id: str) -> Optional[Task]:
        return self._tasks.get(task_id)

    def save(self, task: Task):
        self._tasks[task.id] = task


def _extract_text(message: dict) -> str:
    for p in message.get("parts", []):
        if p.get("kind") == "text":
            return p.get("text", "")
    return ""


def build_agent_server(card: AgentCard, skill_handler):
    """skill_handler must be: async def skill_handler(text: str) -> dict"""
    app = FastAPI(title=card.name)
    store = InMemoryTaskStore()

    @app.get("/.well-known/agent-card.json")
    def get_card():
        return card.model_dump(by_alias=True)

    async def _run_skill(task: Task):
        task.status = TaskState.working
        store.save(task)
        input_text = _extract_text(task.history[-1].model_dump())
        result = await skill_handler(input_text)
        task.artifacts.append(Artifact(name="result", parts=[Part(kind="data", data=result)]))
        task.status = TaskState.completed
        store.save(task)
        return task

    @app.post("/")
    async def rpc(request: Request):
        body = await request.json()
        method, params, req_id = body.get("method"), body.get("params", {}), body.get("id")

        if method == "message/send":
            msg = Message(**params["message"])
            task = store.create(context_id=str(uuid.uuid4())[:8], first_message=msg)
            task = await _run_skill(task)
            return {"jsonrpc": "2.0", "id": req_id, "result": task.model_dump()}

        elif method == "tasks/get":
            task = store.get(params["id"])
            if task is None:
                return JSONResponse(
                    {"jsonrpc": "2.0", "id": req_id, "error": {"code": -32001, "message": "Task not found"}},
                    status_code=404)
            return {"jsonrpc": "2.0", "id": req_id, "result": task.model_dump()}

        elif method == "tasks/cancel":
            task = store.get(params["id"])
            if task and task.status not in TERMINAL_STATES:
                task.status = TaskState.canceled
                store.save(task)
            return {"jsonrpc": "2.0", "id": req_id, "result": {"id": params["id"], "status": task.status if task else None}}

        return JSONResponse(
            {"jsonrpc": "2.0", "id": req_id, "error": {"code": -32601, "message": "Method not found"}},
            status_code=400)

    @app.post("/stream")
    async def rpc_stream(request: Request):
        body = await request.json()
        msg = Message(**body["params"]["message"])
        task = store.create(context_id=str(uuid.uuid4())[:8], first_message=msg)

        async def event_gen():
            yield f"data: {json.dumps({'id': task.id, 'status': TaskState.working.value})}\n\n"
            task.status = TaskState.working
            store.save(task)

            input_text = _extract_text(msg.model_dump())
            result = await skill_handler(input_text)

            task.artifacts.append(Artifact(name="result", parts=[Part(kind="data", data=result)]))
            task.status = TaskState.completed
            store.save(task)
            yield f"data: {json.dumps({'id': task.id, 'artifact': result})}\n\n"
            yield f"data: {json.dumps({'id': task.id, 'status': TaskState.completed.value})}\n\n"

        return StreamingResponse(event_gen(), media_type="text/event-stream")

    return app, store

print("build_agent_server() ready (async skill handlers).")


## 4. Client Helpers (unchanged)


In [ ]:
async def get_client(app: FastAPI, base_url: str = "http://agent.local") -> httpx.AsyncClient:
    transport = httpx.ASGITransport(app=app)
    return httpx.AsyncClient(transport=transport, base_url=base_url)

async def discover(client: httpx.AsyncClient) -> AgentCard:
    r = await client.get("/.well-known/agent-card.json")
    r.raise_for_status()
    return AgentCard(**r.json())

async def send_message(client: httpx.AsyncClient, text: str) -> dict:
    payload = {"jsonrpc": "2.0", "id": str(uuid.uuid4())[:8], "method": "message/send",
               "params": {"message": {"role": "user", "parts": [{"kind": "text", "text": text}]}}}
    r = await client.post("/", json=payload)
    r.raise_for_status()
    return r.json()["result"]

print("Client helpers ready.")


## 5. The LLM-Backed Refund Agent

Instead of regex-ing digits out of a string, the skill handler now genuinely
reads the request and returns structured JSON — the model does the
extraction and reasoning; `extract_json()` handles it if the model wraps its
answer in prose anyway.


In [ ]:
REFUND_SYSTEM_PROMPT = '''You are a refund-processing assistant.
Given a customer's message, extract the order id and decide a plausible refund amount.
Respond with ONLY compact JSON in exactly this shape, no prose, no markdown fences:
{"order_id": "<digits or unknown>", "refund_status": "issued", "amount_usd": <number>}'''

async def refund_skill(order_text: str) -> dict:
    raw = await call_groq(REFUND_SYSTEM_PROMPT, order_text)
    return extract_json(raw)

refund_card = AgentCard(
    name="refund-agent",
    description="Validates and issues refunds for e-commerce orders (LLM-backed)",
    url="https://agents.acme.co/refund",
    version="2.0.0",
    capabilities=AgentCapabilities(streaming=True, push_notifications=False),
    skills=[AgentSkill(id="process-refund", description="Issues a refund for an order id")],
)

refund_app, refund_store = build_agent_server(refund_card, refund_skill)
print("LLM-backed refund-agent is live (in-process).")


In [ ]:
async def demo_llm_refund():
    client = await get_client(refund_app)
    card = await discover(client)
    print(f"Discovered: {card.name} v{card.version}\n")

    result = await send_message(client, "Hey, my order #4471 arrived broken, I'd like a refund please")
    print("Status    :", result["status"])
    print("Artifacts :", result["artifacts"])
    await client.aclose()

asyncio.run(demo_llm_refund())


## 6. Watching the Model Think — Real Token Streaming Through A2A SSE

This is the fun one: Groq streams tokens back to us as they're generated,
and we relay each token straight through the A2A `message/stream` SSE
connection. A client watching this doesn't just see `working → completed` —
it sees the actual answer being written, live.


In [ ]:
async def stream_message(client: httpx.AsyncClient, path: str, text: str):
    payload = {"jsonrpc": "2.0", "id": str(uuid.uuid4())[:8], "method": "message/stream",
               "params": {"message": {"role": "user", "parts": [{"kind": "text", "text": text}]}}}
    async with client.stream("POST", path, json=payload) as resp:
        async for line in resp.aiter_lines():
            if line.startswith("data:"):
                yield json.loads(line[len("data:"):].strip())


def build_streaming_agent_server(card: AgentCard, system_prompt: str):
    """A variant server whose /stream endpoint relays live Groq tokens
    instead of waiting for the full completion."""
    app = FastAPI(title=card.name)
    store = InMemoryTaskStore()

    @app.get("/.well-known/agent-card.json")
    def get_card():
        return card.model_dump(by_alias=True)

    @app.post("/stream")
    async def rpc_stream(request: Request):
        body = await request.json()
        msg = Message(**body["params"]["message"])
        task = store.create(context_id=str(uuid.uuid4())[:8], first_message=msg)
        input_text = _extract_text(msg.model_dump())

        async def event_gen():
            yield f"data: {json.dumps({'id': task.id, 'status': TaskState.working.value})}\n\n"
            task.status = TaskState.working
            store.save(task)

            stream = await groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "system", "content": system_prompt},
                          {"role": "user", "content": input_text}],
                temperature=0.3,
                stream=True,
            )
            accumulated = ""
            async for chunk in stream:
                delta = chunk.choices[0].delta.content or ""
                if delta:
                    accumulated += delta
                    yield f"data: {json.dumps({'id': task.id, 'token': delta})}\n\n"

            task.artifacts.append(Artifact(name="result", parts=[Part(kind="text", text=accumulated)]))
            task.status = TaskState.completed
            store.save(task)
            yield f"data: {json.dumps({'id': task.id, 'status': TaskState.completed.value})}\n\n"

        return StreamingResponse(event_gen(), media_type="text/event-stream")

    return app, store


explainer_card = AgentCard(
    name="explainer-agent", description="Explains things simply, streamed live",
    url="https://agents.acme.co/explainer", version="1.0.0",
    capabilities=AgentCapabilities(streaming=True),
    skills=[AgentSkill(id="explain", description="Explains a concept in plain language")],
)
explainer_app, _ = build_streaming_agent_server(
    explainer_card, "Explain the user's question in 2-3 short sentences, plain language, Feynman-style."
)


async def demo_token_streaming():
    client = await get_client(explainer_app)
    print("Streaming live tokens:\n")
    full_text = ""
    async for event in stream_message(client, "/stream", "Why is the A2A protocol useful?"):
        if "token" in event:
            print(event["token"], end="", flush=True)
            full_text += event["token"]
        elif "status" in event:
            print(f"\n\n[status: {event['status']}]")
    await client.aclose()

asyncio.run(demo_token_streaming())


## 7. Multi-Agent Orchestration — Two Real LLM Agents in Parallel

A **summarizer agent** and a **sentiment agent**, both Groq-backed, both
independent A2A servers. An orchestrator sends the same customer review to
both at once and merges the results — the same fan-out pattern from the
protocol notebook, except every leaf now does genuine reasoning instead of
returning a canned dict.


In [ ]:
async def summarizer_skill(text: str) -> dict:
    summary = await call_groq(
        "Summarize the user's text in one concise sentence. Reply with ONLY that sentence.",
        text, temperature=0.3,
    )
    return {"skill": "summarize", "summary": summary.strip()}

async def sentiment_skill(text: str) -> dict:
    label = await call_groq(
        "Classify the sentiment of the user's text as exactly one word: positive, negative, or neutral. "
        "Reply with ONLY that one word.",
        text, temperature=0.0,
    )
    return {"skill": "sentiment", "label": label.strip().lower()}

summarizer_card = AgentCard(
    name="summarizer-agent", description="Summarizes text", url="https://agents.acme.co/summarizer",
    version="1.0.0", capabilities=AgentCapabilities(streaming=False),
    skills=[AgentSkill(id="summarize", description="Summarizes a passage of text")],
)
sentiment_card = AgentCard(
    name="sentiment-agent", description="Classifies sentiment", url="https://agents.acme.co/sentiment",
    version="1.0.0", capabilities=AgentCapabilities(streaming=False),
    skills=[AgentSkill(id="classify-sentiment", description="Classifies text sentiment")],
)

summarizer_app, _ = build_agent_server(summarizer_card, summarizer_skill)
sentiment_app, _ = build_agent_server(sentiment_card, sentiment_skill)


async def call_agent(app: FastAPI, text: str) -> dict:
    client = await get_client(app)
    try:
        result = await send_message(client, text)
        return result["artifacts"][0]["parts"][0]["data"]
    finally:
        await client.aclose()


async def orchestrate(review_text: str) -> dict:
    summary, sentiment = await asyncio.gather(
        call_agent(summarizer_app, review_text),
        call_agent(sentiment_app, review_text),
    )
    return {"review": review_text, "summary": summary, "sentiment": sentiment}


review = (
    "I ordered the wireless keyboard expecting it to be mediocre based on the price, "
    "but it's honestly fantastic — great battery life, and support replaced a missing "
    "key cap within a day when I reached out. Really impressed."
)
final = asyncio.run(orchestrate(review))
print(json.dumps(final, indent=2))


## 8. Auth Layer (unchanged mechanism, now guarding a real LLM agent)


In [ ]:
VALID_TOKENS = {"demo-token-123"}

def build_secure_agent_server(card: AgentCard, skill_handler):
    app, store = build_agent_server(card, skill_handler)

    @app.middleware("http")
    async def check_auth(request: Request, call_next):
        if request.url.path == "/.well-known/agent-card.json":
            return await call_next(request)
        token = request.headers.get("authorization", "").removeprefix("Bearer ").strip()
        if token not in VALID_TOKENS:
            return JSONResponse({"error": "unauthorized"}, status_code=401)
        return await call_next(request)

    return app, store

secure_refund_card = refund_card.model_copy(update={
    "security_schemes": {"bearer": {"type": "http", "scheme": "bearer"}}
})
secure_refund_app, _ = build_secure_agent_server(secure_refund_card, refund_skill)


async def demo_auth():
    client = await get_client(secure_refund_app)

    r = await client.post("/", json={"jsonrpc": "2.0", "id": "1", "method": "message/send",
                                      "params": {"message": {"role": "user", "parts": [{"kind": "text", "text": "test"}]}}})
    print("No token    ->", r.status_code, r.json())

    r = await client.post("/", headers={"Authorization": "Bearer demo-token-123"},
                           json={"jsonrpc": "2.0", "id": "2", "method": "message/send",
                                 "params": {"message": {"role": "user", "parts": [{"kind": "text", "text": "Refund order #77"}]}}})
    print("Valid token ->", r.status_code, r.json()["result"]["status"])
    await client.aclose()

asyncio.run(demo_auth())


## 9. Production Notes — Specific to LLM-Backed Agents

Everything from the protocol notebook's checklist still applies. On top of
that, LLM-backed skills add their own failure modes:

| # | Concern | What to actually do |
|---|---|---|
| 1 | **Never hardcode API keys in shared/committed notebooks** | This notebook's key cell is for personal, local use only — use `getpass`, a `.env` (gitignored), or a secrets manager for anything shared |
| 2 | **Non-deterministic output** | An LLM skill can return malformed JSON even with a strict prompt — `extract_json()`'s fallback is a minimum, not a full solution; consider `response_format={"type":"json_object"}` where supported |
| 3 | **Timeouts** | Network calls to Groq can hang — set explicit client timeouts, don't let a task sit in `working` forever |
| 4 | **Retry with idempotency** | Same rule as the protocol notebook — a retried `message/send` must not re-charge/re-call the LLM and double-bill both Groq and your own downstream action |
| 5 | **Model deprecation** | Groq retires models on a schedule (see `console.groq.com/docs/deprecations`) — pin a model string, but monitor for deprecation notices |
| 6 | **Cost control per skill** | Cap `max_tokens`, log token usage per task, and consider a cheaper model for high-volume/low-stakes skills, reserving larger models for complex reasoning |
| 7 | **Prompt injection** | Treat the caller's message as untrusted input to the system prompt boundary — don't let it override your instructions or leak your prompt |


## Recap

Same five nouns, same JSON-RPC surface, same task lifecycle — the only thing
that changed between this notebook and the last one is what's *inside* the
skill handler: a static dict became `await call_groq(...)`. That's the whole
point of A2A's opacity principle in practice — the orchestrator, the client,
the auth layer, and the wire format never had to know or care that the
"process-refund" skill started being backed by an LLM instead of hardcoded logic.
